In [ ]:
%pip install torch torchvision numpy pandas scikit-learn

In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

torch.cuda.is_available()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
from torchvision import datasets, transforms

# Augmentation for Training
train_transform = transforms.Compose([
    # Rotate by up to 10 degrees
    transforms.RandomRotation(10), 
    # Randomly shift the image by 10% and scale by 10%
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# No Augmentation for Testing (Standard Preprocessing Only)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load MNIST training and test datasets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=train_transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=test_transform)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")

# Simple CNN Model (~1M params)
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        
        self.conv5 = nn.Conv2d(256, 512, 3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)
        
        self.conv6 = nn.Conv2d(512, 512, 3, padding=1)
        self.bn6 = nn.BatchNorm2d(512)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.4)
        
        self.fc1 = nn.Linear(512 * 3 * 3, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 10)
        
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x)))) # 28 -> 14
        x = self.pool(self.relu(self.bn2(self.conv2(x)))) # 14 -> 7
        x = self.pool(self.relu(self.bn3(self.conv3(x)))) # 7 -> 3
        x = self.relu(self.bn4(self.conv4(x))) # 3 -> 3 (no pool)
        x = self.relu(self.bn5(self.conv5(x))) # 3 -> 3
        x = self.relu(self.bn6(self.conv6(x))) # 3 -> 3
        
        x = x.view(-1, 512 * 3 * 3)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.fc3(x)
        return x

# Initialize model, loss, optimizer
model = MNISTNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training function
def train(model, train_loader, criterion, optimizer, epochs=5):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
            
            if batch_idx % 200 == 0:
                print(f'Epoch {epoch+1}: Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}')
        
        print(f'Epoch {epoch+1} - Loss: {total_loss/len(train_loader):.4f}, Accuracy: {100.*correct/total:.2f}%')

# Evaluation function
def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)

            output = model(data)
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
    
    accuracy = 100. * correct / total
    print(f'Test Accuracy: {accuracy:.2f}%')
    return accuracy

# Train the model
print("Training...")
train(model, train_loader, criterion, optimizer, epochs=10)

# Evaluate
print("\nEvaluating...")
evaluate(model, test_loader)

In [ ]:
# Save the trained model
torch.save(model.state_dict(), 'mnist_model.pth')
print("Model saved to mnist_model.pth")

In [ ]:
# Float16 compression (reliable, works with all models)
model_compressed = MNISTNet().to(device)
model_compressed.load_state_dict(torch.load('mnist_model.pth'))
model_compressed.eval()

# Convert to float16
model_compressed = model_compressed.half()

# Save compressed model
torch.save(model_compressed.state_dict(), 'mnist_model_fp16.pth')

# Clean up failed quantization file
import os
try:
    os.remove('mnist_model_quantized.pth')
except:
    pass

original_size = os.path.getsize('mnist_model.pth') / 1024 / 1024
compressed_size = os.path.getsize('mnist_model_fp16.pth') / 1024 / 1024
print(f"Original model: {original_size:.2f} MB")
print(f"Float16 model: {compressed_size:.2f} MB")
print(f"Size reduction: {100*(1-compressed_size/original_size):.1f}%")

In [ ]:
%pip install torchao

In [ ]:
# Verify fp16 model accuracy on CUDA
model_fp16 = MNISTNet().to(device).half()
model_fp16.load_state_dict(torch.load('mnist_model_fp16.pth', map_location=device))
model_fp16.eval()

# Float16 evaluation function
def evaluate_fp16(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device).half(), target.to(device)
            output = model(data)
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
    accuracy = 100. * correct / total
    print(f'Test Accuracy: {accuracy:.2f}%')
    return accuracy

print("Original model accuracy (CUDA):")
evaluate(model_original, test_loader)

print("\nFloat16 model accuracy (CUDA):")
evaluate_fp16(model_fp16, test_loader)